# Cover detection — GPU training + submission

Thin Colab notebook: all logic lives in the repo (the `covers/` package); this
notebook just stages files, extracts data and calls `covers.train`.

**Before running** put the following in one Google Drive folder:

1. `covers/` (this package)
2. `requirements.txt` (repo root)
3. `train.zip` and `test.zip` (dataset)

Then set `DRIVE_DIR` below to that folder and run all cells. Runtime → *Change
runtime type* → GPU (T4 is enough; A100 nicer).

**What each cell does:** stage files → mount Drive → attach package + deps →
extract CQTs → verify data → train with early stopping on val nDCG@100 → write
`submission.txt` to Drive.

In [ ]:
# ============================ CONFIG ============================
# Google Drive folder containing:  covers/  requirements.txt  train.zip  test.zip
DRIVE_DIR = "/content/drive/MyDrive/ColabNotebooks/Covers"   # <-- CHANGE THIS

# Training options (tweak freely).
TRAIN_EPOCHS     = 25
PATIENCE         = 4      # early stop if val nDCG@100 stalls this many epochs
LR               = 1e-4
BATCH_SIZE       = 128    # reduce to 64/32 if you hit CUDA OOM
MIXED_PRECISION  = True   # keep True on T4/V100 (AMP)
NUM_WORKERS      = 2      # DataLoader workers (0 if Colab complains)
RESUME_CKPT      = None   # e.g. "/content/work/checkpoints/best-model.pt" to resume
# ================================================================

import os
assert os.path.isdir(DRIVE_DIR), f"Drive folder not found: {DRIVE_DIR} (edit DRIVE_DIR above)"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys, shutil, subprocess

CONTENT = "/content"
WORK    = os.path.join(CONTENT, "work")
os.makedirs(WORK, exist_ok=True)

# Attach the covers package from Drive.
shutil.copytree(os.path.join(DRIVE_DIR, "covers"),
                os.path.join(WORK, "covers"), dirs_exist_ok=True)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

# Install pure-python deps (torch reused from Colab; requirement is >=2.4).
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", os.path.join(DRIVE_DIR, "requirements.txt")])

import torch
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import zipfile, time

DATA = os.path.join(WORK, "dataset")   # extracted CQTs: DATA/train, DATA/test

def extract_if_needed(zip_fn: str, dest: str):
    if os.path.isdir(dest) and any(os.scandir(dest)):
        print(f"skip extract: {zip_fn} already present")
        return
    zf = os.path.join(DRIVE_DIR, zip_fn)
    t0 = time.time()
    with zipfile.ZipFile(zf) as z:
        total = len(z.infolist())
        for i, m in enumerate(z.infolist()):
            z.extract(m, dest)
            if i % 20000 == 0:
                print(f"  {zip_fn}: {i/total*100:5.1f}%", flush=True)
    print(f"done {zip_fn} in {time.time()-t0:.0f}s")

extract_if_needed("test.zip",  os.path.join(DATA, "test"))
extract_if_needed("train.zip", os.path.join(DATA, "train"))

In [ ]:
from covers import data as D

train_map, val_map, train_ids, val_ids, ncls = D.train_val_clique_maps(DATA)
print("train cliques:", len(train_map), "| val cliques:", len(val_map),
      "| num_classes:", ncls)
print("train tracks:", len(train_ids), "| val tracks:", len(val_ids))
assert ncls == 39535, f"unexpected num_classes {ncls}"

In [ ]:
from covers.train import TrainCfg, run

cfg = TrainCfg(
    data_path=DATA,
    dataset_path=os.path.join(DATA, "train"),
    num_classes=ncls,
    device="cuda",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    epochs=TRAIN_EPOCHS,
    lr=LR,
    patience=PATIENCE,
    mixed_precision=MIXED_PRECISION,
    checkpoint_dir=os.path.join(WORK, "checkpoints"),
    resume=RESUME_CKPT,
)

# ~1.5h/epoch on a T4 (128 batch). Early stop typically fires well before
# TRAIN_EPOCHS; best weights are kept at <checkpoint_dir>/best-model.pt.
model = run(cfg)   # returns model with the best val nDCG@100 weights loaded

In [ ]:
from covers.train import infer_test

cfg_test = TrainCfg(data_path=DATA,
                    dataset_path=os.path.join(DATA, "train"),
                    num_classes=ncls,
                    device="cuda")
SUB = os.path.join(WORK, "submission.txt")
infer_test(model, cfg_test, submission_path=SUB, batch_size=256)
shutil.copy(SUB, os.path.join(DRIVE_DIR, "submission.txt"))
print("submission saved to Drive:", os.path.join(DRIVE_DIR, "submission.txt"))

## Optional: score the trained model on the val split

Reports `nDCG@100` and mean reciprocal rank over all 15,885 val queries.
CPU-baseline reference: **val nDCG@100 = 0.1660** (mean+std CQT descriptors).

In [ ]:
from covers.dataset import make_dataloaders
from covers.train import compute_val_score, build_track_to_clique

loaders = make_dataloaders(DATA, os.path.join(DATA, "train"), train_map,
                           train_ids=[], val_ids=val_ids, test_ids=None,
                           batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
track_to_clique = build_track_to_clique(val_map, val_ids)
print(compute_val_score(model, loaders["val"], torch.device("cuda"),
                        track_to_clique, k=100))

## Notes

- Training progress + per-epoch metrics are appended to
  `work/checkpoints/history.jsonl`; best weights live in
  `work/checkpoints/best-model.pt`.
- Session restarts are safe: files persist in Drive, and re-running the
  extraction cell skips existing data. Restart *from cell 4* after Dataset is
  reattached; set `RESUME_CKPT` and `TRAIN_EPOCHS` to continue.
- If you hit a name clash after uploading a fixed `covers/` from Drive,
  restart the runtime and re-run the cells — Python caches the module.
- The local CPU baseline equivalent:  `python -m covers.baseline_cpu
  --dataset_path dataset\dataset --mode val`.